# Perbandingan Evaluasi HPO dan K-Fold Cross Validation

Notebook ini digunakan untuk mengagregasi, membandingkan, dan memvisualisasikan hasil dari 6 model YOLO dan RT-DETR yang dilatih secara paralel pada 6 akun cloud yang berbeda.

Notebook ini akan:
1. Memuat berkas metrik evaluasi (`.csv`) dari masing-masing model.
2. Memuat berkas hyperparameter terbaik (`.json`) hasil pencarian algoritma HPO.
3. Menghitung rata-rata metrik evaluasi dari pengujian 5-Fold Cross Validation.
4. Membuat visualisasi perbandingan metrik mAP50 dalam bentuk grafik batang.
5. Visualisasi stabilitas model menggunakan boxplot sebaran mAP50 di seluruh fold.
6. Memformat tabel perbandingan agar siap dimasukkan ke dalam buku Tugas Akhir (format Markdown & LaTeX).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pengaturan visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 16
})
print("Library berhasil diimpor dan visualisasi siap digunakan.")

## 1. Konfigurasi Nama Model dan File Output

Di sini kita mendefinisikan model-model yang digunakan dan nama file output metrik hasil training dari masing-masing notebook.

In [ ]:
models = ['yolov8n', 'yolov9t', 'yolov10n', 'yolo11n', 'yolo26n', 'rtdetr-l']

# Kamus pemetaan file CSV metrik dan JSON parameter
csv_files = {m: f"final_metrics_comparison_{m}.csv" for m in models}
json_files = {m: f"hpo_best_parameters_{m}.json" for m in models}

print("Daftar file metrik yang dicari:")
for k, v in csv_files.items():
    print(f"- Model {k.upper()}: {v}")

## 2. Simulasi Data Awal (Opsional)

Jika Anda menjalankan notebook ini sebelum seluruh training paralel di cloud selesai, cell berikut akan mendeteksi file metrik yang belum diunduh dan membuat data simulasi secara otomatis. Hal ini berguna untuk memvalidasi alur grafik dan kode visualisasi sebelum data asli terkumpul.

In [ ]:
def generate_mock_csv(model_name):
    # Menentukan nilai seed berdasarkan model name agar data simulasi konsisten
    np.random.seed(42 + hash(model_name) % 100)
    methods = ['Baseline', 'PSO', 'GWO', 'WOA', 'ACO']
    folds = [0, 1, 2, 3, 4]
    
    # Perkiraan nilai mAP awal baseline
    base_performance = {
        'yolov8n': 0.782,
        'yolov9t': 0.795,
        'yolov10n': 0.774,
        'yolo11n': 0.803,
        'yolo26n': 0.811,
        'rtdetr-l': 0.765
    }.get(model_name, 0.75)

    # Perkiraan rata-rata peningkatan mAP50 dari optimasi HPO
    lift = {
        'Baseline': 0.0,
        'PSO': 0.042,
        'GWO': 0.051,
        'WOA': 0.048,
        'ACO': 0.034
    }

    rows = []
    for method in methods:
        target_map = base_performance + lift[method]
        for fold in folds:
            # Variabilitas fold-to-fold acak
            variation = np.random.normal(0, 0.015)
            map50 = min(0.99, max(0.5, target_map + variation))
            map50_95 = map50 * 0.71 + np.random.normal(0, 0.008)
            precision = min(0.99, max(0.5, map50 * 1.04 + np.random.normal(0, 0.015)))
            recall = min(0.99, max(0.5, map50 * 0.96 + np.random.normal(0, 0.015)))
            
            rows.append({
                'Method': method,
                'Fold': fold,
                'Precision': round(precision, 4),
                'Recall': round(recall, 4),
                'mAP50': round(map50, 4),
                'mAP50-95': round(map50_95, 4)
            })
            
    df = pd.DataFrame(rows)
    df.to_csv(f"final_metrics_comparison_{model_name}.csv", index=False)
    print(f"-> Simulasi metrik dibuat untuk {model_name.upper()}")

def generate_mock_json(model_name):
    # Hyperparameter acak yang mendekati rentang parameter pencarian
    np.random.seed(42 + hash(model_name) % 100)
    mock_params = {}
    for method in ['PSO', 'GWO', 'WOA', 'ACO']:
        mock_params[method] = {
            "lr0": round(np.random.uniform(0.005, 0.06), 5),
            "lrf": round(np.random.uniform(0.01, 0.15), 5),
            "momentum": round(np.random.uniform(0.8, 0.95), 5),
            "weight_decay": round(np.random.uniform(0.0002, 0.0015), 5),
            "mosaic": round(np.random.uniform(0.6, 1.0), 4),
            "box": round(np.random.uniform(6.0, 9.5), 4)
        }
    with open(f"hpo_best_parameters_{model_name}.json", "w") as f:
        json.dump(mock_params, f, indent=4)
    print(f"-> Simulasi json parameter dibuat untuk {model_name.upper()}")

# Jalankan pengecekan file
for m in models:
    if not os.path.exists(csv_files[m]):
        generate_mock_csv(m)
    if not os.path.exists(json_files[m]):
        generate_mock_json(m)

## 3. Penggabungan Berkas Metrik Hasil Training

Pada tahapan ini, data metrik dari seluruh model digabungkan ke dalam satu DataFrame utama (`df_master`).

In [ ]:
all_dfs = []
for m in models:
    df = pd.read_csv(csv_files[m])
    df['Model'] = m.upper()
    all_dfs.append(df)

df_master = pd.concat(all_dfs, ignore_index=True)
print(f"Metrik dari {len(models)} model berhasil digabungkan.")
print(f"Total baris data evaluasi: {len(df_master)}")
df_master.head(10)

## 4. Perhitungan Rata-Rata Metrik Evaluasi (5-Fold)

Metrik dihitung rata-ratanya dari seluruh fold (Fold 0-4) untuk mendapatkan gambaran performa umum model.

In [ ]:
# Mengelompokkan data berdasarkan Model dan Metode, kemudian menghitung nilai rata-rata
df_summary = df_master.groupby(['Model', 'Method']).mean().reset_index()
df_summary = df_summary.drop(columns=['Fold'])

# Simpan hasil agregasi tabel ke file CSV lokal
df_summary.to_csv("aggregated_models_summary.csv", index=False)

# Format tampilan tabel agar mudah dibaca di notebook
df_styled = df_summary.copy()
for col in ['Precision', 'Recall', 'mAP50', 'mAP50-95']:
    df_styled[col] = df_styled[col].map(lambda x: f"{x:.4f}")

df_styled

## 5. Grafik Perbandingan Kinerja (mAP50)

Grafik ini memvisualisasikan perbandingan performa mAP50 rata-rata di antara baseline model dengan hasil optimasi hyperparameter menggunakan PSO, GWO, WOA, dan ACO.

In [ ]:
plt.figure(figsize=(15, 8))
ax = sns.barplot(
    x='Model', 
    y='mAP50', 
    hue='Method', 
    data=df_summary, 
    palette='viridis'
)

# Menambahkan judul dan label
plt.title('Perbandingan Kinerja mAP50 Baseline vs Hasil Optimasi HPO (Mean 5-Fold)', pad=20)
plt.xlabel('Model Arsitektur')
plt.ylabel('mAP50')
plt.ylim(0.70, 0.95)
plt.legend(title='Metode', bbox_to_anchor=(1.02, 1), loc='upper left')

# Menampilkan angka nilai di atas batang grafik
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(
            f"{p.get_height():.3f}", 
            (p.get_x() + p.get_width() / 2., p.get_height()), 
            ha='center', va='center', 
            xytext=(0, 8), 
            textcoords='offset points', 
            fontsize=9, 
            rotation=90
        )

plt.tight_layout()
plt.savefig('comparison_map50_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Analisis Boxplot Stabilitas Performa 5-Fold

Boxplot di bawah menunjukkan sebaran mAP50 di seluruh fold. Boxplot yang memiliki rentang tinggi yang kecil/sempit mengindikasikan bahwa model tersebut sangat stabil dan tangguh terhadap variasi partisi data latih/uji.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 20), sharey=True)
axes = axes.flatten()

for i, m in enumerate(models):
    df_sub = df_master[df_master['Model'] == m.upper()]
    sns.boxplot(
        x='Method', 
        y='mAP50', 
        data=df_sub, 
        ax=axes[i], 
        palette='pastel'
    )
    sns.stripplot(
        x='Method', 
        y='mAP50', 
        data=df_sub, 
        ax=axes[i], 
        color='black', 
        alpha=0.3, 
        jitter=0.15
    )
    axes[i].set_title(f"Stabilitas Kinerja 5-Fold: {m.upper()}")
    axes[i].set_xlabel("Metode")
    axes[i].set_ylabel("mAP50")

plt.suptitle('Analisis Stabilitas Performa mAP50 pada 5-Fold Cross Validation', y=1.01)
plt.tight_layout()
plt.savefig('stability_boxplot_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Analisis Perbandingan Hyperparameter Hasil Pencarian HPO

Berikut adalah tabel perbandingan set hyperparameter terbaik yang ditemukan oleh masing-masing algoritma metaheuristik untuk seluruh model.

In [ ]:
all_params = []
for m in models:
    with open(json_files[m], 'r') as f:
        params = json.load(f)
    for method, hyp in params.items():
        hyp_row = hyp.copy()
        hyp_row['Model'] = m.upper()
        hyp_row['Method'] = method
        all_params.append(hyp_row)

df_params = pd.DataFrame(all_params)
# Menyusun ulang urutan kolom
cols = ['Model', 'Method'] + [c for c in df_params.columns if c not in ['Model', 'Method']]
df_params = df_params[cols]

df_params.to_csv("all_models_best_hyperparameters.csv", index=False)
print("Tabel Hyperparameter Hasil Optimasi HPO:")
df_params

## 8. Export Hasil ke LaTeX / Markdown untuk Buku Tugas Akhir

Cell ini akan menghasilkan string format LaTeX dan Markdown yang siap disalin langsung ke dalam naskah bab 4 tulisan Tugas Akhir Anda.

In [ ]:
# Membuat tabel pivot mAP50
pivot_map50 = df_summary.pivot(index='Model', columns='Method', values='mAP50')
print("=== TABEL MARKDOWN (mAP50) ===\n")
print(pivot_map50.to_markdown())
print("\n" + "="*40 + "\n")

print("=== KODE LATEX (mAP50) ===\n")
print(pivot_map50.to_latex(float_format="%.4f"))